# ZS601 v3：E 组 LiDAR 深度监督（L4，150000 步）

E = C 的几何约束与硬尺度上限 + 遮挡感知 LiDAR camera-z 深度 loss。数据先从 Drive 复制到 Colab 本地。正式训练前为 train/val/test 生成 16-bit 伪 GT 深度图并写入 Drive，再从保存后的 PNG 反投影到 3D 验证。任一相机覆盖率或反投影不合格时，训练不会开始。

固定 val 每 5000 步输出 RGB、法向、深度和 1σ 彩色椭球；不保存 geometry.npz。checkpoint/PLY 每 50000 步保存。最终对完整 test 集计算指标并保存最差 10 个相机的四类诊断图。请在 Colab 中选择 **Tesla L4**。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import csv, hashlib, json, os, shutil, subprocess, sys, uuid, zipfile
import numpy as np
import torch

CODE_REF='b3d2137600dadccfa949d16d4aecee916dca2b1c'
REPO_URL='https://github.com/VISjudy/ZS601_3DGS.git'
BRANCH='v3-e'

DRIVE_DATA_ROOT=Path('/content/drive/MyDrive/LCCDataset/ZS601meetingroom')
DATA_ZIP=DRIVE_DATA_ROOT/'ZS601meetingroom_data.zip'
SPLIT_ROOT=Path('/content/drive/MyDrive/LCCDataset/zs601_output/gaussian-splattingWithMask_v3_cff221ccfb')
VAL_SOURCE=SPLIT_ROOT/'images-val10.txt'
TEST_SOURCE=SPLIT_ROOT/'images_test.txt'
RESULTS_ROOT=Path('/content/drive/MyDrive/LCCDataset/zs601_output')

RUN_ID='zs601_E_'+uuid.uuid4().hex[:10]
WORK=Path('/content')/RUN_ID
RESULTS=RESULTS_ROOT/RUN_ID
DEPTH_EXPORT=DRIVE_DATA_ROOT/('lidar_depth_gt_E_'+RUN_ID)
WORK.mkdir()
RESULTS.mkdir()
assert not DEPTH_EXPORT.exists(), DEPTH_EXPORT

ITERATIONS=150000
COMMON={
    'sh_degree':2,
    'position_lr_init':0.000016,
    'position_lr_final':0.00000016,
    'position_lr_max_steps':150000,
    'scaling_lr':0.0015,
    'seed':42,
    'lazy_cache':100,
}
DEPTH={
    'lambda_lidar_depth':0.05,
    'lidar_depth_start':1000,
    'lidar_depth_warmup':4000,
    'lidar_depth_min':0.1,
    'lidar_depth_max':15.0,
    'lidar_depth_splat_radius':1,
    'lidar_depth_edge_relative':0.02,
    'lidar_depth_edge_absolute':0.02,
    'lidar_depth_min_neighbors':2,
    'lidar_depth_alpha_min':0.05,
    'lidar_depth_min_pixels':64,
    'lidar_depth_min_coverage':0.001,
    'lidar_depth_huber_beta':0.02,
    'lidar_depth_distance_power':1.0,
    'lidar_depth_weight_min':0.25,
    'lidar_depth_weight_max':4.0,
    'lidar_depth_backproject_samples':4096,
    'lidar_depth_backproject_tolerance':0.06,
    'lidar_depth_backproject_quantile':0.99,
    'lidar_depth_backproject_min_fraction':0.99,
    'lidar_depth_reprojection_tolerance_px':2.0,
}
OVERRIDES={}  # 示例：{'normal_loss':'off'}；每项仅用同名 on/off 覆盖。
AUTO_RELEASE_RUNTIME=True

def run(args,cwd=None):
    args=list(map(str,args))
    print('COMMAND',args,flush=True)
    logfile=RESULTS/('command_'+uuid.uuid4().hex[:10]+'.log')
    with logfile.open('x',encoding='utf-8') as log:
        with subprocess.Popen(args,cwd=cwd,stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
            for line in process.stdout:
                log.write(line)
                log.flush()
                if not line.startswith('Reading camera'):
                    print(line,end='',flush=True)
            returncode=process.wait()
    if returncode:
        raise subprocess.CalledProcessError(returncode,args)
    print('LOG',logfile)

assert torch.cuda.is_available(), 'CUDA GPU unavailable'
gpu=torch.cuda.get_device_name(0)
assert 'L4' in gpu, f'Request Tesla L4, got {gpu}'
run(['nvidia-smi'])
print({'python':sys.version,'torch':torch.__version__,'cuda':torch.version.cuda,'gpu':gpu})
run(['git','clone','--branch',BRANCH,REPO_URL,WORK/'repo'])
run(['git','checkout','--detach',CODE_REF],cwd=WORK/'repo')
CODE=WORK/'repo/gaussian-splattingWithMask_v3'
assert CODE.is_dir()


## 环境构建与 CPU 合同测试

保留 Colab 自带 PyTorch。测试覆盖几何约束、尺度限制、z-buffer 最近深度、遮挡边缘补洞、像素中心、相机变换、uint16 深度编码和距离权重。随后编译并导入 CUDA 扩展。

In [ ]:
run([sys.executable,'-m','pip','install',
     'ninja','plyfile','laspy','scipy','pillow',
     'opencv-python-headless','tqdm','cupy-cuda12x'])
os.environ['MAX_JOBS']='2'
for name in ['simple-knn','diff-gaussian-rasterization']:
    run([sys.executable,'-m','pip','install','-v','--no-build-isolation',
         CODE/'submodules'/name])
run([sys.executable,'-c',
     'import torch,cupy; from simple_knn._C import distCUDA2; '
     'from diff_gaussian_rasterization import GaussianRasterizer; '
     'print(torch.cuda.get_device_name(0),cupy.__version__)'],cwd=CODE)
run([sys.executable,'-m','unittest','-v',
     'test_geometry_v3','test_scale_bounds_v3','test_lidar_depth_v3'],cwd=CODE)


## 将数据复制到 Colab 本地并建立固定划分

原 ZIP、相机列表和 mask 只读。训练列表写在本次临时目录，排除固定 val/test；不读取可能绕过划分的 images.bin。

In [ ]:
for source in [DATA_ZIP,VAL_SOURCE,TEST_SOURCE]:
    assert source.is_file(), source
LOCAL_ZIP=WORK/'data.zip'
shutil.copy2(DATA_ZIP,LOCAL_ZIP)
DATA=WORK/'data'
DATA.mkdir()
with zipfile.ZipFile(LOCAL_ZIP) as archive:
    for item in archive.infolist():
        target=(DATA/item.filename).resolve()
        assert target.is_relative_to(DATA.resolve()), item.filename
        assert (item.external_attr>>16)&0o170000!=0o120000, 'ZIP symlink'
    archive.extractall(DATA)

POINTS=DATA/'ZS601_3cm_sample.las'
INTR=DATA/'sparse/cameras.txt'
POSES=DATA/'sparse/images.txt'
VAL=WORK/'images-val10.txt'
TEST=WORK/'images_test.txt'
TRAIN=WORK/'images_train_v3.txt'
shutil.copy2(VAL_SOURCE,VAL)
shutil.copy2(TEST_SOURCE,TEST)
for source in [POINTS,INTR,POSES]:
    assert source.is_file(), source
run([sys.executable,'prepare_v3.py',
     '--images_file',POSES,'--val_file',VAL,'--test_file',TEST,
     '--output_train',TRAIN],cwd=CODE)
for source in [VAL,TEST,TRAIN]:
    shutil.copy2(source,RESULTS/source.name)
print({'local_data':str(DATA),'point_cloud':str(POINTS),
       'train_file':str(TRAIN),'val_file':str(VAL),'test_file':str(TEST),
       'depth_export':str(DEPTH_EXPORT)})


In [ ]:
def train_command(out,iterations,cache,depth_export=None,resume=None,
                  depth_overrides=None,final_test='on'):
    command=[
        sys.executable,'train_mask_v3.py','--experiment','E',
        '-s',DATA,'-m',out,'--point_cloud',POINTS,
        '--train_file',TRAIN,'--val_file',VAL,'--test_file',TEST,
        '--cameras_file',INTR,'--units','scene',
        '--iterations',iterations,'--val_interval',5000,
        '--checkpoint_interval',50000,'--val_npz','off',
        '--val_ellipsoids','on','--final_test',final_test,
        '--lidar_depth_cache',cache,
    ]
    if depth_export is not None:
        command += ['--lidar_depth_export',depth_export]
    settings={**COMMON,**DEPTH,**(depth_overrides or {})}
    for key,value in settings.items():
        command += ['--'+key,str(value)]
    for key,value in OVERRIDES.items():
        command += ['--'+key,str(value)]
    if resume is not None:
        command += ['--resume',resume]
    return command

def verify_run(out,steps):
    done=json.loads((out/'completed.json').read_text(encoding='utf-8'))
    assert done['iteration']==steps,done
    loss=list(csv.DictReader((out/'loss_log.csv').open(encoding='utf-8')))
    assert loss
    first=int(loss[0]['iteration'])-1
    assert [int(row['iteration']) for row in loss]==list(range(first+1,steps+1))
    assert all(np.isfinite(float(row['total'])) for row in loss)
    active=[row for row in loss if row['lidar_depth_state']=='ACTIVE']
    assert active,'LiDAR depth loss never became ACTIVE'
    assert max(int(float(row['lidar_depth_valid_pixels'])) for row in active)>=DEPTH['lidar_depth_min_pixels']
    assert all(np.isfinite(float(row['lidar_depth_distance_weighted_raw'])) for row in active)

    val=list(csv.DictReader((out/'val_metrics.csv').open(encoding='utf-8')))
    expected=(([0] if first==0 else [])+
              [iteration for iteration in range(first+1,steps+1)
               if iteration%5000==0 or iteration==steps])
    assert len(val)==11*len(expected),(len(val),expected)
    for iteration in expected:
        directory=out/'val_v3'/f'iteration_{iteration:06d}'
        for kind in ['rgb','normal','depth','ellipsoid']:
            assert len(list(directory.glob('*_'+kind+'.png')))==10,(directory,kind)
        assert not list(directory.glob('*_geometry.npz'))
    assert (out/'checkpoints'/f'iteration_{steps}.pth').is_file()
    print('VERIFIED RUN',out,{'first':first,'steps':steps,
          'loss_rows':len(loss),'depth_active_rows':len(active),'val_rows':len(val)})
    return {'first':first,'steps':steps,'loss_rows':len(loss),
            'depth_active_rows':len(active),'val_rows':len(val)}


## 200 步 E 冒烟

冒烟将深度 warmup 提前，用来确认 CUDA 深度属性渲染、有效 mask、距离权重、反向传播、CSV 和固定 val 输出。冒烟不导出全量 Drive 深度数据，也不代表视觉质量。

In [ ]:
SMOKE_OUT=RESULTS/'smoke_E'
SMOKE_CACHE=WORK/'lidar_depth_cache_smoke'
run(train_command(
    SMOKE_OUT,200,SMOKE_CACHE,
    depth_overrides={'lidar_depth_start':0,'lidar_depth_warmup':50},
    final_test='off',
),cwd=CODE)
SMOKE_VERIFY=verify_run(SMOKE_OUT,200)


## 正式 E 实验：150000 步

正式命令先把所有 train/val/test LiDAR 伪深度保存到新的 Drive 数据集目录。它重新读取每张 uint16 PNG，以像素中心反投影回 3D，检查到原始 LiDAR 的 Q99 最近邻距离、通过率和重投影像素误差；全部通过后才创建高斯并训练。

如果运行时中断，可在新 Colab 运行时重新执行前面的单元，将 `RESUME_CHECKPOINT` 指向完整 checkpoint，并将 `DEPTH_EXPORT` 改为已有且 `verification.json` 通过的深度目录。模型输出仍须使用新目录；已验证深度目录只读复用。

In [ ]:
FORMAL_OUT=RESULTS/'formal_E'
FORMAL_CACHE=WORK/'lidar_depth_cache_formal'
RESUME_CHECKPOINT=None
run(train_command(
    FORMAL_OUT,ITERATIONS,FORMAL_CACHE,
    depth_export=DEPTH_EXPORT,
    resume=RESUME_CHECKPOINT,
    final_test='on',
),cwd=CODE)
FORMAL_VERIFY=verify_run(FORMAL_OUT,ITERATIONS)


## Drive 产物验收与释放运行时

验收伪深度数据集、完整 test 指标、最差 10 个相机的 RGB/1σ 椭球/深度/法向图、最终总结和 50000 步 checkpoint 规则。所有检查通过并执行 `os.sync()` 后才释放 Colab。

In [ ]:
depth_summary=json.loads((DEPTH_EXPORT/'dataset_summary.json').read_text(encoding='utf-8'))
depth_verification=json.loads((DEPTH_EXPORT/'verification.json').read_text(encoding='utf-8'))
depth_rows=list(csv.DictReader((DEPTH_EXPORT/'depth_manifest.csv').open(encoding='utf-8')))
assert depth_verification['passed'] is True
assert depth_verification['checked_saved_png_round_trip'] is True
assert depth_verification['checked_camera_to_world_backprojection'] is True
assert len(depth_rows)==depth_summary['camera_count']
assert len(list(DEPTH_EXPORT.rglob('*_depth_u16.png')))==depth_summary['camera_count']
expected_previews=(depth_summary['roles']['val']['camera_count']+
                   depth_summary['roles']['test']['camera_count'])
assert len(list(DEPTH_EXPORT.rglob('*_depth_color.png')))==expected_previews
assert not list(DEPTH_EXPORT.rglob('*.npz'))
assert depth_summary['backprojection']['all_cameras_passed'] is True

completed=json.loads((FORMAL_OUT/'completed.json').read_text(encoding='utf-8'))
assert completed['final_test_complete'] is True
test_dir=FORMAL_OUT/'test_final'/'iteration_150000'
test_summary=json.loads((test_dir/'test_summary.json').read_text(encoding='utf-8'))
test_rows=list(csv.DictReader((test_dir/'test_metrics.csv').open(encoding='utf-8')))
assert len(test_rows)==test_summary['camera_count']+4
assert len(test_summary['worst10'])==min(10,test_summary['camera_count'])
for row in test_summary['worst10']:
    for filename in row['files'].values():
        assert (test_dir/filename).is_file(),filename
assert (FORMAL_OUT/'experiment_summary.md').is_file()
assert not list((FORMAL_OUT/'val_v3').rglob('*_geometry.npz'))
assert not list(test_dir.rglob('*.npz'))
for iteration in [50000,100000,150000]:
    assert (FORMAL_OUT/'checkpoints'/f'iteration_{iteration}.pth').is_file()
    assert (FORMAL_OUT/'point_cloud'/f'iteration_{iteration}'/'point_cloud.ply').is_file()

workflow={
    'passed':True,'code_ref':CODE_REF,'branch':BRANCH,'gpu':gpu,
    'smoke':SMOKE_VERIFY,'formal':FORMAL_VERIFY,
    'result':str(FORMAL_OUT),'depth_dataset':str(DEPTH_EXPORT),
    'depth_verification':depth_verification,
    'test_mean':test_summary['aggregates']['mean'],
}
with (RESULTS/'workflow_verified.json').open('x',encoding='utf-8') as handle:
    json.dump(workflow,handle,indent=2)
os.sync()
print('E COMPLETE AND VERIFIED',json.dumps(workflow,indent=2))

if AUTO_RELEASE_RUNTIME:
    from google.colab import runtime
    runtime.unassign()
